In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_percentage_error

In [15]:
df = pd.read_csv("data/house_dataset.csv")
df.tail(1)

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
9199,2014-07-10 00:00:00,220600.0,3.0,2.5,1490,8102,2.0,0,0,4,1490,0,1990,0,18717 SE 258th St,Covington,WA 98042,USA


## preprocessing

In [16]:
# Convert date to datetime
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Very simple date feature engineering
df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month

# Drop original date column
df = df.drop(columns=["date"])

# Quick check
print(df.shape)
print(df.isnull().sum())

(9200, 19)
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
street           0
city             0
statezip         0
country          0
sale_year        0
sale_month       0
dtype: int64


## train/test split 80/20

In [5]:
X = df.drop(columns=["price"])
y = df["price"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (7360, 18)
Test shape: (1840, 18)


In [7]:
numeric_features = [
    "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
    "waterfront", "view", "condition", "sqft_above", "sqft_basement",
    "yr_built", "yr_renovated", "sale_year", "sale_month"
]

categorical_features = ["street", "city", "statezip", "country"]

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'sale_year', 'sale_month']
Categorical: ['street', 'city', 'statezip', 'country']


In [8]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [9]:
models = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.001, max_iter=10000),
    "ElasticNet": ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000)
}

In [ ]:
# target = price
results = []

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    
    # Avoid division issues if price were ever zero
    preds = np.maximum(preds, 1)
    
    mape = mean_absolute_percentage_error(y_test, preds)
    
    results.append({
        "model": name,
        "target_type": "raw_price",
        "MAPE": mape
    })

results_df = pd.DataFrame(results).sort_values("MAPE")
results_df

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:656: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.817e+13, tolerance: 1.875e+11
  model = cd_fast.sparse_enet_coordinate_descent(


,model,target_type,MAPE
2,Lasso,raw_price,5.494620e+18
0,OLS,raw_price,6.353336e+18
1,Ridge,raw_price,2.746599e+19
3,ElasticNet,raw_price,3.938804e+19


In [11]:
results_log = []

y_train_log = np.log1p(y_train)

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    pipe.fit(X_train, y_train_log)
    
    log_preds = pipe.predict(X_test)
    preds = np.expm1(log_preds)   # convert back to normal price scale
    preds = np.maximum(preds, 1)
    
    mape = mean_absolute_percentage_error(y_test, preds)
    
    results_log.append({
        "model": name,
        "target_type": "log_price",
        "MAPE": mape
    })

results_log_df = pd.DataFrame(results_log).sort_values("MAPE")
results_log_df

,model,target_type,MAPE
1,Ridge,log_price,4.725524e+18
0,OLS,log_price,4.796006e+18
2,Lasso,log_price,6.908973e+18
3,ElasticNet,log_price,8.961489e+18


In [13]:
all_results = pd.concat([results_df, results_log_df], ignore_index=True)
all_results = all_results.sort_values("MAPE")
best_row = all_results.iloc[0]
print("Best model:")
print(best_row)

Best model:
model                          Ridge
target_type                log_price
MAPE           4725524369454452736.0
Name: 4, dtype: object


## xgboost baseline

In [14]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

xgb_pipe.fit(X_train, y_train)
xgb_preds = xgb_pipe.predict(X_test)
xgb_preds = np.maximum(xgb_preds, 1)

xgb_mape = mean_absolute_percentage_error(y_test, xgb_preds)
print("XGBoost MAPE:", xgb_mape)

XGBoost MAPE: 3.5880170964923027e+19
